In [2]:
%pip install matplotlib==3.7.3

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 6.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 6.0 MB/s  0:00:03 eta 0:00:01
    sys-platform (=="darwin") ; extra == 'objc'
                 ~^
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: matplotlib━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: matplotlib 3.10.80/2 [numpy]
    Uninstalling matplotlib-3.10.8:━━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled matplotlib-3.10.8━━━━━━━━━━━━━━━━━━ 1/2 [matplotlib]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [matplotlib]2 [matplotlib]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install "numpy<2" --force-reinstall

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
    sys-platform (=="darwin") ; extra == 'objc'
                 ~^
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. 파일 경로 설정 (사용자 환경에 맞게 수정하세요)
# ---------------------------------------------------------
ARFF_PATH = "/home/junhyung/study/Data_Analysis_with_CKKS/Cluster/DBSCAN_CKKS/desilo/dataset/Other_cluster/hepta.arff"
CSV_PATH = "/home/junhyung/study/Data_Analysis_with_CKKS/Cluster/DBSCAN_CKKS/desilo/hepta_fhe_heap_result_eps1.0_min4.csv" 

def load_arff_to_pts(filepath):
    pts = []
    data_section = False
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('%'): continue
            if line.lower().startswith('@data'):
                data_section = True
                continue
            if data_section:
                line = line.replace('\t', ' ').replace(',', ' ')
                values = line.split()
                if len(values) < 2: continue
                # 마지막 라벨을 제외한 X, Y, Z 좌표만 추출
                row = [float(v) for v in values[:-1]]
                pts.append(row)
    return np.array(pts, dtype=np.float64)

def main():
    print("데이터 및 결과 파일 로딩 중...")
    
    # 2. 데이터 로드
    pts = load_arff_to_pts(ARFF_PATH)
    df = pd.read_csv(CSV_PATH)
    
    if len(pts) != len(df):
        print("🚨 경고: ARFF 데이터 개수와 CSV 결과 개수가 다릅니다!")
        return

    # X, Y, Z 좌표 분리 (Hepta는 3차원)
    X = pts[:, 0]
    Y = pts[:, 1]
    Z = pts[:, 2]
    
    pt_labels = df['True_Class'].values
    fhe_labels = df['FHE_Cluster'].values

    # 3. 3D 시각화 설정
    fig = plt.figure(figsize=(16, 8))
    
    # 노이즈(-1)는 검은색(k) 별표(x)로, 나머지는 컬러맵 적용
    cmap = plt.get_cmap('tab20') # 20가지 색상을 지원하는 컬러맵

    # --- [왼쪽 그래프: Plaintext 결과] ---
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.set_title("Plaintext DBSCAN (Expected)", fontsize=16, fontweight='bold')
    
    for label in np.unique(pt_labels):
        idx = pt_labels == label
        if label == -1:
            ax1.scatter(X[idx], Y[idx], Z[idx], c='black', marker='x', s=20, label='Noise(-1)')
        else:
            ax1.scatter(X[idx], Y[idx], Z[idx], cmap=cmap, s=50, alpha=0.8, edgecolors='w')

    ax1.set_xlabel('X axis')
    ax1.set_ylabel('Y axis')
    ax1.set_zlabel('Z axis')

    # --- [오른쪽 그래프: FHE 결과] ---
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.set_title("FHE DBSCAN (ARI: 49.57)", fontsize=16, fontweight='bold')
    
    for label in np.unique(fhe_labels):
        idx = fhe_labels == label
        if label == -1:
            ax2.scatter(X[idx], Y[idx], Z[idx], c='black', marker='x', s=20, label='Noise(-1)')
        else:
            ax2.scatter(X[idx], Y[idx], Z[idx], cmap=cmap, s=50, alpha=0.8, edgecolors='w')

    ax2.set_xlabel('X axis')
    ax2.set_ylabel('Y axis')
    ax2.set_zlabel('Z axis')

    # 4. 출력 및 저장
    plt.tight_layout()
    
    # 이미지로 저장 (보고서/논문용)
    save_filename = "hepta_comparison_plot.png"
    plt.savefig(save_filename, dpi=300)
    print(f"✅ 그래프가 '{save_filename}' 로 저장되었습니다.")
    
    # 화면에 띄우기 (GUI 환경인 경우 마우스로 돌려볼 수 있습니다)
    plt.show()

if __name__ == '__main__':
    main()


데이터 및 결과 파일 로딩 중...


FileNotFoundError: [Errno 2] No such file or directory: '/home/junhyung/study/Data_Analysis_with_CKKS/Cluster/DBSCAN_CKKS/desilo/hepta_fhe_result_eps1.0_min4.0.csv'

In [ ]:
import time
import numpy as np
import pytest
from desilofhe import Engine
from util.cipher_packer import CipherPacker
from util.data import KeyPack
from util.packer import packer, unpacker

# =====================================================================
# [중요] 도출된 다항식 계수 (기함수 형태: y = 0.5 + c1*x + c3*x^3 + c5*x^5)
# FHE 내부 연산 함수 구현 시 이 계수들을 multiply_plain 에 사용하십시오.
# =====================================================================
COEFS = {
    "Legendre_Deg3":  {"c1": 0.249562, "c3": -0.018723, "c5": 0.0},
    "Chebyshev_Deg3": {"c1": 0.249432, "c3": -0.018480, "c5": 0.0},
    "Remez_Deg3":     {"c1": 0.249440, "c3": -0.018491, "c5": 0.0},
    
    "Legendre_Deg5":  {"c1": 0.249986, "c3": -0.020701, "c5": 0.001780},
    "Chebyshev_Deg5": {"c1": 0.249981, "c3": -0.020677, "c5": 0.001757},
    "Remez_Deg5":     {"c1": 0.249981, "c3": -0.020678, "c5": 0.001758},
}

# (가정) 사용자님의 FHE 다항식 연산 모듈에서 3차/5차 함수들을 임포트한다고 가정합니다.
# 실제 구현 시 위 COEFS 딕셔너리의 값을 사용하여 x, x^3, x^5 를 더해주는 로직을 구현하셔야 합니다.
from fhe.core.sigmoid import (
    UnitLegendreDeg3Sigmoid_HE, LegendreDeg3Sigmoid_HE,
    UnitChebyshevDeg3Sigmoid_HE, ChebyshevDeg3Sigmoid_HE,
    UnitRemezOddDeg3Sigmoid_HE, RemezOddDeg3Sigmoid_HE,
    UnitLegendreDeg5Sigmoid_HE, LegendreDeg5Sigmoid_HE,
    UnitChebyshevDeg5Sigmoid_HE, ChebyshevDeg5Sigmoid_HE,
    UnitRemezOddDeg5Sigmoid_HE, RemezOddDeg5Sigmoid_HE
)

def true_sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

# =====================================================================
# Pytest Fixtures
# =====================================================================
@pytest.fixture(scope="module")
def engine():
    return Engine(use_bootstrap=True, mode="gpu")

@pytest.fixture(scope="module")
def secret_key(engine):
    return engine.create_secret_key()

@pytest.fixture(scope="module")
def keypack(engine, secret_key):
    keypack = KeyPack(
        engine.create_public_key(secret_key),
        None,
        engine.create_relinearization_key(secret_key),
        engine.create_conjugation_key(secret_key),
        engine.create_lossy_bootstrap_key(secret_key, stage_count=3),
    )
    return keypack

@pytest.fixture(scope="module")
def cipher_packer(engine):
    return CipherPacker(engine)

# =====================================================================
# 테스트 파라미터 (요청하신 3가지 Range 적용)
# =====================================================================
RANGES = [
    (-1.0, 1.0),
    (-1.2, 1.2),
    (-1.5, 1.5),
]

UNIT_TEST_CASES = [
    (1, 4096, 1 << 15),
]

TENSOR_TEST_CASES = [
    ((4096, 64), 1 << 15),
]

# 3차, 5차 함수 매핑
TARGET_FUNCTIONS = [
    ("Legendre_Deg3", UnitLegendreDeg3Sigmoid_HE, LegendreDeg3Sigmoid_HE),
    ("Chebyshev_Deg3", UnitChebyshevDeg3Sigmoid_HE, ChebyshevDeg3Sigmoid_HE),
    ("Remez_Deg3", UnitRemezOddDeg3Sigmoid_HE, RemezOddDeg3Sigmoid_HE),
    ("Legendre_Deg5", UnitLegendreDeg5Sigmoid_HE, LegendreDeg5Sigmoid_HE),
    ("Chebyshev_Deg5", UnitChebyshevDeg5Sigmoid_HE, ChebyshevDeg5Sigmoid_HE),
    ("Remez_Deg5", UnitRemezOddDeg5Sigmoid_HE, RemezOddDeg5Sigmoid_HE)
]

# =====================================================================
# 데이터 생성 헬퍼 함수
# =====================================================================
def build_base_signal(signal_size, low, high):
    np.random.seed(42) # 비교 일관성을 위해 시드 고정
    base = np.random.uniform(low, high, 20).astype(np.float64)
    base.sort()
    return np.resize(base, signal_size)

def build_unit_signal(batch, signal_size, low, high):
    tiled = build_base_signal(signal_size, low, high)
    signal = []
    for _ in range(batch):
        signal.append(tiled.copy())
    return signal

def build_tensor_signal(signal_size, batch, low, high):
    tiled = build_base_signal(signal_size, low, high)
    signal = np.zeros((signal_size, batch), dtype=np.float64)
    for i in range(batch):
        signal[:, i] = tiled
    return signal

# =====================================================================
# [UNIT TEST] 20개 점 테이블 출력 및 시간 측정
# =====================================================================
@pytest.mark.parametrize("low, high", RANGES, scope="function")
@pytest.mark.parametrize("batch, signal_size, slot_counts", UNIT_TEST_CASES, scope="function")
def test_Unit_Sigmoid_Functions(engine, secret_key, keypack, batch, signal_size, slot_counts, low, high):
    public_key = keypack.public_key
    signal = build_unit_signal(batch, signal_size, low, high)
    gap = slot_counts // signal_size
    packed_signal = packer(signal, gap, slot_counts)
    
    ciphertext = engine.encrypt(packed_signal, public_key)
    x_20 = signal[0][:20]
    true_y_20 = true_sigmoid(x_20)
    
    REPEAT_TIME = 5
    print(f"\n==================== HE vs TRUE SIGMOID ON [{low}, {high}] ====================")
    print(f"repeat for mean eval time: {REPEAT_TIME}")

    for func_name, unit_he_func, _ in TARGET_FUNCTIONS:
        eval_times = []
        
        for _ in range(REPEAT_TIME):
            start_time = time.time()
            output_ct = unit_he_func(ciphertext, keypack, engine)
            eval_times.append(time.time() - start_time)
            
        mean_eval_time = np.mean(eval_times)
        
        plaintext = engine.decrypt(output_ct, secret_key)
        unpacked_output = unpacker(plaintext, gap, batch)
        he_y_20 = unpacked_output[0][:20]
        
        abs_errors = np.abs(true_y_20 - he_y_20)
        max_err = np.max(abs_errors)
        mean_err = np.mean(abs_errors)
        
        # 1.5 구간 외삽의 경우 오차가 최대 0.006~0.01 근처까지 갈 수 있으므로 atol을 0.02로 약간 여유있게 둡니다.
        is_close = np.allclose(true_y_20, he_y_20, atol=2e-2, rtol=1e-3)

        print(f"\n===== {func_name} =====")
        print("idx |          x | true sigmoid |  he output |      abs err")
        print("---------------------------------------------------------------")
        for i in range(20):
            print(f"{i:3d} | {x_20[i]:10.6f} |   {true_y_20[i]:10.8f} | {he_y_20[i]:10.8f} | {abs_errors[i]:12.8e}")
        
        print(f"\n{func_name} vs true sigmoid")
        print(f"{func_name} np.allclose(atol=2e-2, rtol=1e-3): {is_close}")
        print(f"{func_name} max abs err: {max_err:.16e}")
        print(f"{func_name} mean abs err: {mean_err:.16e}")
        print(f"{func_name} mean eval time (sec): {mean_eval_time:.6f}")
        
        assert is_close, f"{func_name} failed np.allclose assertion against True Sigmoid at range [{low}, {high}]!"

# =====================================================================
# [TENSOR TEST] 대규모 병렬 텐서 연산
# =====================================================================
@pytest.mark.parametrize("low, high", RANGES, scope="function")
@pytest.mark.parametrize("input_shape, slot_counts", TENSOR_TEST_CASES, scope="function")
def test_Tensor_Sigmoid_Functions(engine, secret_key, keypack, cipher_packer, input_shape, slot_counts, low, high):
    signal_size, batch = input_shape
    signal = build_tensor_signal(signal_size, batch, low, high)
    
    true_y_tensor = true_sigmoid(signal)
    
    public_key = keypack.public_key
    cipherTensor = cipher_packer.pack(signal, public_key)

    for func_name, _, tensor_he_func in TARGET_FUNCTIONS:
        outputCipherTensor = tensor_he_func(cipherTensor, keypack, engine)
        he_output_tensor = cipher_packer.unpack(outputCipherTensor, secret_key)
        
        # 텐서 단위에서도 1.5 구간을 커버하기 위해 atol=2e-2 적용
        is_close = np.allclose(true_y_tensor, he_output_tensor, atol=2e-2, rtol=1e-3)
        assert is_close, f"Tensor test failed for {func_name} against True Sigmoid at range [{low}, {high}]!"